In [19]:
# 📌 Objectif : Associer un label (1=inondé, 0=non-inondé) à chaque superpixel par vote majoritaire

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, shape
from sklearn.neighbors import KDTree
import numpy as np
import json
import rasterio
from skimage.segmentation import slic

In [23]:


# === 1. Charger les superpixels ===
with rasterio.open("Matam_NDWI_SRTM_2024.tif") as src:
    img = src.read()
    transform = src.transform

img = np.where(np.isnan(img), 0, img)
image_for_slic = np.moveaxis(img, 0, -1)
segments = slic(image_for_slic, n_segments=1000, compactness=10, start_label=1)
np.save("segments.npy", segments)  # 🔁 Sauvegarde des superpixels pour usage futur

ndwi = img[0]
srtm = img[1]
segment_ids = np.unique(segments)
features = []

for seg_id in segment_ids:
    mask = segments == seg_id
    mean_ndwi = ndwi[mask].mean()
    mean_srtm = srtm[mask].mean()
    indices = np.argwhere(mask)
    yx_center = indices.mean(axis=0)
    row, col = int(yx_center[0]), int(yx_center[1])
    x, y = rasterio.transform.xy(transform, row, col)
    features.append({
        'segment_id': seg_id,
        'mean_ndwi': mean_ndwi,
        'mean_srtm': mean_srtm,
        'longitude': x,
        'latitude': y
    })

# === 2. Créer GeoDataFrame superpixels ===
df_superpixels = pd.DataFrame(features)
df_superpixels["geometry"] = df_superpixels.apply(lambda row: Point(row["longitude"], row["latitude"]), axis=1)
gdf_superpixels = gpd.GeoDataFrame(df_superpixels, geometry="geometry", crs="EPSG:4326")

# === 3. Charger les échantillons supervisés ===
samples = pd.read_csv("Samples_NDWI_SRTM_Matam_Final_3500.csv")
samples["geometry"] = samples[".geo"].apply(lambda g: shape(json.loads(g)))
samples["lon"] = samples["geometry"].apply(lambda p: p.x)
samples["lat"] = samples["geometry"].apply(lambda p: p.y)
gdf_samples = gpd.GeoDataFrame(samples, geometry="geometry", crs="EPSG:4326")

# === 4. Associer chaque point à son superpixel le plus proche ===
tree = KDTree(gdf_superpixels[["longitude", "latitude"]].values)
distances, indices = tree.query(gdf_samples[["lon", "lat"]].values, k=1)
gdf_samples["superpixel_id"] = indices.flatten()

# === 5. Vote majoritaire par superpixel ===
label_votes = gdf_samples.groupby("superpixel_id")["classvalue"].agg(lambda x: 1 if x.mean() >= 0.5 else 0).reset_index()
label_votes.columns = ["segment_id", "label"]

# === 6. Fusion avec les features superpixels ===
df_final = pd.merge(df_superpixels, label_votes, on="segment_id", how="left")
df_final = df_final.dropna(subset=["label"])  # supprimer les superpixels non couverts

df_final.to_csv("superpixels_with_labels.csv", index=False)
print("✅ superpixels_with_labels.csv exporté avec succès :", df_final.shape[0], "échantillons")


✅ superpixels_with_labels.csv exporté avec succès : 900 échantillons


In [25]:
# 📌 Recréer segments.npy de manière stable à partir du raster NDWI + SRTM d'origine

import rasterio
import numpy as np
from skimage.segmentation import slic

# === 1. Charger le raster NDWI + SRTM ===
raster_path = 'Matam_NDWI_SRTM_2024.tif'
with rasterio.open(raster_path) as src:
    img = src.read()  # (bands, height, width)

# === 2. Nettoyer les NaN pour éviter les erreurs SLIC ===
img = np.where(np.isnan(img), 0, img)

# === 3. Reformatage pour skimage (H, W, C) ===
image_for_slic = np.moveaxis(img, 0, -1)

# === 4. Appliquer la segmentation SLIC de manière stable ===
segments = slic(
    image_for_slic,
    n_segments=1000,
    compactness=10,
    start_label=1,
    channel_axis=-1,
    enforce_connectivity=True
)

# === 5. Sauvegarder segments.npy ===
np.save("segments.npy", segments)
print("✅ Segments SLIC sauvegardés dans 'segments.npy' avec", len(np.unique(segments)), "superpixels")


✅ Segments SLIC sauvegardés dans 'segments.npy' avec 990 superpixels
